# Классификаторы и Регрессия scikit-learn
## Применение к данным стоматологической клиники

**Задание 5**  
**ВКР:** «Информационная система стоматологической клиники»  
**Платформа:** Jupyter Notebook


---

### Разница: классификация vs регрессия

| Задача | Выход | Пример |
|---|---|---|
| **Классификация** | Дискретная метка класса | Тип лечения: профилактика / кариес / пульпит |
| **Регрессия** | Непрерывное число | Стоимость приёма (₽) |

### Рассматриваемые методы:
1. **Logistic Regression** — логистическая регрессия (классификатор)
2. **Linear Regression** — линейная регрессия
3. **Decision Tree** — дерево решений
4. **Naive Bayes** — наивный байесовский классификатор
5. **Random Forest** — случайный лес
6. **Gradient Boosting (XGBoost-style)** — градиентный бустинг
7. **SVM** — метод опорных векторов
8. **k-NN** — k ближайших соседей
9. **MLP (Neural Network)** — многослойный перцептрон
10. **Сравнение и оценка** — метрики, кросс-валидация, ROC-кривые

## ⚙️ 0. Установка и импорт

In [ ]:
# !pip install scikit-learn matplotlib seaborn numpy pandas -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Классификаторы ────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text, plot_tree
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor,
    AdaBoostClassifier
)
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor

# ── Метрики и утилиты ────────────────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    GridSearchCV, learning_curve, StratifiedKFold
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, auc, roc_auc_score,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.multiclass import OneVsRestClassifier

# Настройка графиков
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'DejaVu Sans'
PALETTE = ['#2563EB','#16A34A','#DC2626','#D97706','#7C3AED',
           '#0D9488','#DB2777','#EA580C','#65A30D','#6B7280']

print('✅ Все библиотеки импортированы')
import sklearn; print(f'   scikit-learn {sklearn.__version__}')

## 🗂️ 1. Датасет стоматологической клиники

In [ ]:
np.random.seed(42)
N = 600

# ── Признаки ──────────────────────────────────────────────────────────────────
# Класс 0: Профилактика (1 зуб, дешево, молодые)
# Класс 1: Кариес/Пломба (2-3 зуба, средне)
# Класс 2: Пульпит/Сложное (3-5 зубов, дорого, старше)

data = []
for cls, (n, tc_mu, tc_sig, cost_mu, cost_sig, age_mu, age_sig, mat_mu) in enumerate([
    (200, 1.2, 0.4, 1500, 400, 28, 8, 1.2),   # 0: профилактика
    (230, 2.5, 0.6, 5500, 1200, 38, 12, 2.8), # 1: кариес
    (170, 3.8, 0.7, 11000, 2500, 52, 15, 4.1),# 2: пульпит/сложное
]):
    teeth   = np.clip(np.random.normal(tc_mu, tc_sig, n), 1, 8)
    cost    = np.clip(np.random.normal(cost_mu, cost_sig, n), 500, 20000)
    age     = np.clip(np.random.normal(age_mu, age_sig, n), 18, 85)
    materials = np.clip(np.random.normal(mat_mu, 0.8, n), 1, 8).round().astype(int)
    visits  = np.clip(np.random.poisson(cls + 1, n), 1, 15)
    duration = np.clip(cost / 1000 + np.random.normal(0, 0.5, n), 0.5, 5)
    for i in range(n):
        data.append({'teeth_count': round(teeth[i],1),
                     'cost': round(cost[i],0),
                     'age': int(age[i]),
                     'materials_count': materials[i],
                     'visits': visits[i],
                     'duration_h': round(duration[i],2),
                     'treatment_type': cls})

df = pd.DataFrame(data)
CLASS_NAMES = ['Профилактика', 'Кариес/Пломба', 'Пульпит/Сложное']

# Признаки и метки
FEATURES = ['teeth_count', 'cost', 'age', 'materials_count', 'visits', 'duration_h']
X_raw = df[FEATURES].values
y     = df['treatment_type'].values
y_cost = df['cost'].values  # для регрессии

# Нормализация
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

# Train/Test split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y_cost, test_size=0.25, random_state=42)

print(f'Датасет: {len(df)} записей, {len(FEATURES)} признаков')
print(f'Train: {len(X_tr)}, Test: {len(X_te)}')
print(f'\nРаспределение классов:')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {i} — {name}: {(y==i).sum()} записей')

# ── EDA ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.flat, FEATURES):
    for cls, name, color in zip([0,1,2], CLASS_NAMES, PALETTE):
        vals = df[df.treatment_type==cls][feat]
        ax.hist(vals, bins=20, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(feat, fontweight='bold', fontsize=11)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
plt.suptitle('EDA: распределение признаков по классам лечения', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Корреляционная матрица
fig, ax = plt.subplots(figsize=(8, 5))
corr = df[FEATURES + ['treatment_type']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlBu_r', center=0,
            ax=ax, square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Корреляционная матрица признаков', fontweight='bold')
plt.tight_layout(); plt.show()

---
## Метод 1: Логистическая регрессия (Logistic Regression)

### Задача
Классификация типа лечения (0/1/2) по признакам приёма.

### Математика
Для бинарной задачи логистическая функция (сигмоид) преобразует линейную комбинацию признаков в вероятность:

$$\hat{p} = \sigma(\mathbf{w}^T \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^T \mathbf{x} + b)}}$$

Для многоклассовой задачи — **Softmax**:
$$P(y=k|\mathbf{x}) = \frac{e^{\mathbf{w}_k^T \mathbf{x}}}{\sum_{j=1}^{K} e^{\mathbf{w}_j^T \mathbf{x}}}$$

**Функция потерь** — кросс-энтропия (Log Loss):
$$L = -\frac{1}{N}\sum_{i=1}^{N} y_i \log \hat{p}_i + (1-y_i) \log(1-\hat{p}_i)$$

**Регуляризация:** L2 (Ridge): $L + \frac{\lambda}{2}||\mathbf{w}||^2$  |  L1 (Lasso): $L + \lambda||\mathbf{w}||_1$

### Параметры:
- `C` — обратная сила регуляризации (1/λ). Мал → сильная регуляризация
- `penalty` — тип регуляризации: `'l2'`, `'l1'`, `'elasticnet'`, `'none'`
- `solver` — оптимизатор: `'lbfgs'`, `'saga'`, `'liblinear'`
- `multi_class='multinomial'` — softmax для многоклассовой задачи

In [ ]:
# ── Логистическая регрессия ───────────────────────────────────────────────────
lr = LogisticRegression(C=1.0, penalty='l2', solver='lbfgs',
                        multi_class='multinomial', max_iter=500, random_state=42)
lr.fit(X_tr, y_tr)
y_pred_lr = lr.predict(X_te)
y_prob_lr = lr.predict_proba(X_te)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Матрица ошибок
cm = confusion_matrix(y_te, y_pred_lr)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0].set_title('Матрица ошибок (Confusion Matrix)', fontweight='bold')
axes[0].set_ylabel('Истинный класс'); axes[0].set_xlabel('Предсказанный класс')
axes[0].tick_params(axis='x', rotation=15)

# Коэффициенты модели
coef_df = pd.DataFrame(lr.coef_.T, index=FEATURES, columns=CLASS_NAMES)
coef_df.plot(kind='bar', ax=axes[1], color=PALETTE[:3], alpha=0.8, edgecolor='white')
axes[1].set_title('Веса признаков (коэффициенты)', fontweight='bold')
axes[1].set_xlabel('Признак'); axes[1].set_ylabel('Вес')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3, axis='y')
axes[1].axhline(0, color='black', lw=0.8)

# Влияние C (регуляризации)
C_vals = [0.001, 0.01, 0.1, 1, 10, 100]
tr_acc, te_acc = [], []
for c in C_vals:
    m = LogisticRegression(C=c, max_iter=500, random_state=42)
    m.fit(X_tr, y_tr)
    tr_acc.append(accuracy_score(y_tr, m.predict(X_tr)))
    te_acc.append(accuracy_score(y_te, m.predict(X_te)))
axes[2].semilogx(C_vals, tr_acc, 'o-', color=PALETTE[0], label='Train', lw=2)
axes[2].semilogx(C_vals, te_acc, 's--', color=PALETTE[1], label='Test', lw=2)
axes[2].axvline(x=1.0, color='red', linestyle=':', alpha=0.7, label='C=1 (выбрано)')
axes[2].set_xlabel('C (обратная регуляризация, log scale)')
axes[2].set_ylabel('Accuracy'); axes[2].set_title('Влияние параметра C', fontweight='bold')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle(' Логистическая регрессия — Стоматологическая клиника', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

acc = accuracy_score(y_te, y_pred_lr)
print(f'\nLogistic Regression результаты:')
print(f'  Accuracy:  {acc:.4f}')
print(f'  F1-macro:  {f1_score(y_te, y_pred_lr, average="macro"):.4f}')
print()
print(classification_report(y_te, y_pred_lr, target_names=CLASS_NAMES))
print(' Интерпретация: положительный коэффициент у "cost" для класса 2 означает,')
print('   что рост стоимости увеличивает вероятность "Пульпит/Сложное".')

---
##  Метод 2: Линейная регрессия (Linear Regression)

### Задача
Предсказание **стоимости приёма** (₽) по признакам — непрерывный выход.

### Математика
Модель: линейная комбинация признаков:
$$\hat{y} = \mathbf{w}^T \mathbf{x} + b = w_1 x_1 + w_2 x_2 + ... + w_n x_n + b$$

**Метод наименьших квадратов (OLS)** — минимизация MSE:
$$\text{MSE} = \frac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2$$

**Аналитическое решение:** $\mathbf{w} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$

**Ridge (L2):** $\min ||y - Xw||^2 + \alpha||w||^2$ — уменьшает коэффициенты  
**Lasso (L1):** $\min ||y - Xw||^2 + \alpha||w||_1$ — обнуляет незначимые признаки

### Метрики регрессии:
- **MAE** = $\frac{1}{N}\sum|y_i - \hat{y}_i|$ — средняя абсолютная ошибка
- **RMSE** = $\sqrt{\text{MSE}}$ — корень из MSE (в единицах целевой переменной)
- **R²** = $1 - \frac{SS_{res}}{SS_{tot}}$ — коэффициент детерминации [0,1]

In [ ]:
# ── Линейная регрессия и её вариации ─────────────────────────────────────────
models_reg = {
    'Linear (OLS)': LinearRegression(),
    'Ridge (L2, α=1)': Ridge(alpha=1.0),
    'Ridge (L2, α=100)': Ridge(alpha=100.0),
    'Lasso (L1, α=10)': Lasso(alpha=10.0, max_iter=5000),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

results_reg = {}
for (name, model), color in zip(models_reg.items(), PALETTE):
    model.fit(X_tr_r, y_tr_r)
    pred = model.predict(X_te_r)
    results_reg[name] = {
        'MAE':  mean_absolute_error(y_te_r, pred),
        'RMSE': np.sqrt(mean_squared_error(y_te_r, pred)),
        'R²':   r2_score(y_te_r, pred),
        'pred': pred,
        'coef': model.coef_,
    }

# Scatter: предсказанное vs истинное (OLS)
pred_ols = results_reg['Linear (OLS)']['pred']
axes[0,0].scatter(y_te_r, pred_ols, alpha=0.5, s=25, color=PALETTE[0], edgecolors='white', lw=0.3)
mn, mx = y_te_r.min(), y_te_r.max()
axes[0,0].plot([mn,mx],[mn,mx],'r--',lw=2,label='Идеал (y=x)')
axes[0,0].set_xlabel('Истинная стоимость (₽)'); axes[0,0].set_ylabel('Предсказанная (₽)')
r2 = results_reg['Linear (OLS)']['R²']
axes[0,0].set_title(f'OLS: Предсказание vs Истина\nR²={r2:.3f}', fontweight='bold')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# Остатки (residuals)
residuals = y_te_r - pred_ols
axes[0,1].scatter(pred_ols, residuals, alpha=0.5, s=25, color=PALETTE[1], edgecolors='white', lw=0.3)
axes[0,1].axhline(0, color='red', linestyle='--', lw=2)
axes[0,1].set_xlabel('Предсказанное'); axes[0,1].set_ylabel('Остаток (y - ŷ)')
axes[0,1].set_title('График остатков (Residuals)', fontweight='bold')
axes[0,1].grid(True, alpha=0.3)

# Коэффициенты моделей
coef_data = {name: res['coef'] for name, res in results_reg.items()}
coef_df_r = pd.DataFrame(coef_data, index=FEATURES)
coef_df_r.plot(kind='bar', ax=axes[0,2], color=PALETTE[:4], alpha=0.8, edgecolor='white')
axes[0,2].set_title('Коэффициенты: OLS vs Ridge vs Lasso', fontweight='bold')
axes[0,2].axhline(0, color='black', lw=0.8)
axes[0,2].tick_params(axis='x', rotation=30)
axes[0,2].legend(fontsize=7); axes[0,2].grid(True, alpha=0.3, axis='y')

# Метрики сравнения
metric_names = ['MAE', 'RMSE', 'R²']
for ax, metric in zip([axes[1,0], axes[1,1], axes[1,2]], metric_names):
    vals = [results_reg[n][metric] for n in models_reg]
    bars = ax.bar(list(models_reg.keys()), vals,
                  color=PALETTE[:4], alpha=0.85, edgecolor='white')
    ax.set_title(f'{metric}', fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
               f'{val:.1f}' if metric!='R²' else f'{val:.3f}',
               ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.suptitle(' Линейная регрессия: предсказание стоимости приёма (₽)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('\nСравнение моделей регрессии:')
print(f'{"Модель":<22} {"MAE":>8} {"RMSE":>8} {"R²":>8}')
print('-'*50)
for name, res in results_reg.items():
    print(f'{name:<22} {res["MAE"]:>8.1f} {res["RMSE"]:>8.1f} {res["R²"]:>8.4f}')
print()
print(' Lasso (L1) обнуляет незначимые коэффициенты — автоматический отбор признаков.')
print('   Ridge (L2) уменьшает все коэффициенты равномерно — устойчивость к выбросам.')

---
## Метод 3: Дерево решений (Decision Tree)

### Математика
Дерево рекурсивно разбивает пространство признаков, выбирая на каждом шаге **наилучшее разбиение** по критерию:

**Критерий Джини (Gini Impurity):**
$$G(t) = 1 - \sum_{k=1}^{K} p_k^2$$
где $p_k$ — доля класса $k$ в узле $t$.

**Информационная выгода (Entropy):**
$$H(t) = -\sum_{k=1}^{K} p_k \log_2 p_k$$
$$IG = H(parent) - \frac{N_{left}}{N}H(left) - \frac{N_{right}}{N}H(right)$$

Выбирается разбиение с **максимальной** информационной выгодой.

### Параметры:
- `max_depth` — максимальная глубина (контроль переобучения)
- `min_samples_split` — минимум точек для разбиения узла
- `min_samples_leaf` — минимум точек в листе
- `criterion` — `'gini'` или `'entropy'`

In [ ]:
# ── Дерево решений: классификация типа лечения ────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Влияние глубины на переобучение
depths = range(1, 12)
tr_scores, te_scores = [], []
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_tr, y_tr)
    tr_scores.append(accuracy_score(y_tr, dt.predict(X_tr)))
    te_scores.append(accuracy_score(y_te, dt.predict(X_te)))

axes[0,0].plot(list(depths), tr_scores, 'o-', color=PALETTE[0], lw=2, label='Train')
axes[0,0].plot(list(depths), te_scores, 's--', color=PALETTE[1], lw=2, label='Test')
best_d = depths[np.argmax(te_scores)]
axes[0,0].axvline(x=best_d, color='red', linestyle=':', label=f'Лучшая глубина={best_d}')
axes[0,0].set_xlabel('max_depth'); axes[0,0].set_ylabel('Accuracy')
axes[0,0].set_title('Влияние глубины дерева', fontweight='bold')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# Лучшее дерево
dt_best = DecisionTreeClassifier(max_depth=best_d, criterion='gini', random_state=42)
dt_best.fit(X_tr, y_tr)
y_pred_dt = dt_best.predict(X_te)

# Матрица ошибок
cm_dt = confusion_matrix(y_te, y_pred_dt)
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens', ax=axes[0,1],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0,1].set_title(f'Матрица ошибок (depth={best_d})', fontweight='bold')
axes[0,1].tick_params(axis='x', rotation=15)

# Важность признаков
fi = dt_best.feature_importances_
sorted_idx = np.argsort(fi)
axes[0,2].barh([FEATURES[i] for i in sorted_idx], fi[sorted_idx],
               color=[PALETTE[i%len(PALETTE)] for i in range(len(FEATURES))], alpha=0.85)
axes[0,2].set_title('Важность признаков (Feature Importance)', fontweight='bold')
axes[0,2].set_xlabel('Importance'); axes[0,2].grid(True, alpha=0.3, axis='x')

# Визуализация дерева
dt_viz = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
dt_viz.fit(X_tr, y_tr)
plot_tree(dt_viz, ax=axes[1,0], feature_names=FEATURES,
          class_names=CLASS_NAMES, filled=True, rounded=True,
          fontsize=7, impurity=True, proportion=False)
axes[1,0].set_title('Визуализация дерева (depth=3)', fontweight='bold')

# Граница решения (2 главных признака)
X2 = X[:, :2]  # teeth_count, cost
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y, test_size=0.25, random_state=42, stratify=y)
dt2 = DecisionTreeClassifier(max_depth=4, random_state=42)
dt2.fit(X2_tr, y2_tr)
h = 0.05
x_min, x_max = X2[:,0].min()-0.5, X2[:,0].max()+0.5
y_min, y_max = X2[:,1].min()-0.5, X2[:,1].max()+0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = dt2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[1,1].contourf(xx, yy, Z, alpha=0.3, cmap='Set1')
for cls in [0,1,2]:
    mask = y2_te == cls
    axes[1,1].scatter(X2_te[mask,0], X2_te[mask,1], c=PALETTE[cls],
                     s=30, alpha=0.8, edgecolors='white', lw=0.3, label=CLASS_NAMES[cls])
axes[1,1].set_title('Граница решения (teeth vs cost)', fontweight='bold')
axes[1,1].set_xlabel('Зубы (норм.)'); axes[1,1].set_ylabel('Стоимость (норм.)')
axes[1,1].legend(fontsize=8); axes[1,1].grid(True, alpha=0.2)

# Текстовое правило дерева
rules = export_text(dt_viz, feature_names=FEATURES)
axes[1,2].text(0.02, 0.98, rules, transform=axes[1,2].transAxes,
               fontsize=7, verticalalignment='top', fontfamily='monospace',
               bbox=dict(boxstyle='round', facecolor='#F1F5F9', alpha=0.8))
axes[1,2].axis('off')
axes[1,2].set_title('Правила дерева (текст)', fontweight='bold')

plt.suptitle('🌳 Дерево решений — Классификация типа лечения', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\nDecision Tree (depth={best_d}):')
print(f'  Accuracy: {accuracy_score(y_te, y_pred_dt):.4f}')
print(f'  F1-macro: {f1_score(y_te, y_pred_dt, average="macro"):.4f}')
print()
print(' Главное преимущество Decision Tree — интерпретируемость.')
print('   Врач может читать правила: "если зубов > 2.5 И стоимость > 8000 → Пульпит".')

---
## Метод 4: Наивный байесовский классификатор (Naive Bayes)

### Математика
Основан на **теореме Байеса:**

$$P(y|\mathbf{x}) = \frac{P(\mathbf{x}|y) \cdot P(y)}{P(\mathbf{x})}$$

**«Наивное» допущение:** признаки независимы при условии класса:
$$P(\mathbf{x}|y) = \prod_{i=1}^{n} P(x_i|y)$$

Поэтому: $\hat{y} = \arg\max_y P(y) \prod_{i=1}^{n} P(x_i|y)$

### Три варианта:
- **GaussianNB** — признаки непрерывные, следуют нормальному распределению:  
  $P(x_i|y) = \frac{1}{\sqrt{2\pi\sigma_{iy}^2}} \exp\left(-\frac{(x_i-\mu_{iy})^2}{2\sigma_{iy}^2}\right)$
- **MultinomialNB** — для счётных данных (частоты слов, количества посещений)
- **BernoulliNB** — для бинарных признаков (есть/нет, True/False)

### Параметр:
- `var_smoothing` (GaussianNB) — сглаживание дисперсии для нулевых вероятностей

In [ ]:
# ── Наивный Байес ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Три варианта NB
X_pos = X - X.min(axis=0) + 1e-6  # позитивные значения для MultinomialNB
X_tr_p, X_te_p, _, _ = train_test_split(X_pos, y, test_size=0.25, random_state=42, stratify=y)

nb_models = {
    'GaussianNB': (GaussianNB(), X_tr, X_te),
    'MultinomialNB': (MultinomialNB(), X_tr_p, X_te_p),
    'BernoulliNB': (BernoulliNB(), X_tr, X_te),
}

nb_results = {}
for name, (model, Xtr, Xte) in nb_models.items():
    model.fit(Xtr, y_tr)
    pred = model.predict(Xte)
    nb_results[name] = {
        'acc': accuracy_score(y_te, pred),
        'f1':  f1_score(y_te, pred, average='macro'),
        'pred': pred, 'model': model,
    }

# Матрица ошибок GaussianNB
cm_gnb = confusion_matrix(y_te, nb_results['GaussianNB']['pred'])
sns.heatmap(cm_gnb, annot=True, fmt='d', cmap='Oranges', ax=axes[0,0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0,0].set_title('GaussianNB: Матрица ошибок', fontweight='bold')
axes[0,0].tick_params(axis='x', rotation=15)

# Апостериорные вероятности (posterior)
gnb = nb_results['GaussianNB']['model']
probs_gnb = gnb.predict_proba(X_te)
sample_idxs = [0, 10, 20]  # несколько примеров
width = 0.25
x_pos = np.arange(3)
for i, (idx, color) in enumerate(zip(sample_idxs, PALETTE[:3])):
    axes[0,1].bar(x_pos + i*width, probs_gnb[idx], width,
                 label=f'Пример {idx} (класс {y_te[idx]})', color=color, alpha=0.8)
axes[0,1].set_xticks(x_pos + width)
axes[0,1].set_xticklabels(CLASS_NAMES, rotation=15, fontsize=8)
axes[0,1].set_ylabel('Вероятность P(y|x)')
axes[0,1].set_title('Апостериорные вероятности GaussianNB', fontweight='bold')
axes[0,1].legend(fontsize=8); axes[0,1].grid(True, alpha=0.3, axis='y')

# Сравнение трёх NB
names_nb = list(nb_results.keys())
accs = [nb_results[n]['acc'] for n in names_nb]
f1s  = [nb_results[n]['f1']  for n in names_nb]
x = np.arange(len(names_nb)); w = 0.35
axes[0,2].bar(x-w/2, accs, w, label='Accuracy', color=PALETTE[0], alpha=0.85)
axes[0,2].bar(x+w/2, f1s, w, label='F1-macro', color=PALETTE[1], alpha=0.85)
axes[0,2].set_xticks(x); axes[0,2].set_xticklabels(names_nb, fontsize=9)
axes[0,2].set_title('Сравнение GaussianNB / MultinomialNB / BernoulliNB', fontweight='bold')
axes[0,2].legend(); axes[0,2].grid(True, alpha=0.3, axis='y')
axes[0,2].set_ylim(0, 1.1)

# Визуализация гауссовских распределений P(xi|y)
for fi_idx, feat_name, ax in zip([1, 0], ['cost (норм.)', 'teeth_count (норм.)'], [axes[1,0], axes[1,1]]):
    x_plot = np.linspace(X[:, fi_idx].min()-0.5, X[:, fi_idx].max()+0.5, 200)
    for cls_idx, (cls_name, color) in enumerate(zip(CLASS_NAMES, PALETTE)):
        mu  = gnb.theta_[cls_idx, fi_idx]
        sig = np.sqrt(gnb.var_[cls_idx, fi_idx])
        gauss = np.exp(-0.5*((x_plot-mu)/sig)**2) / (sig*np.sqrt(2*np.pi))
        ax.plot(x_plot, gauss, color=color, lw=2.5, label=cls_name)
        ax.axvline(mu, color=color, linestyle=':', alpha=0.7, lw=1.5)
    ax.set_title(f'P({feat_name} | класс) — гауссовское ядро', fontweight='bold')
    ax.set_xlabel(feat_name); ax.set_ylabel('Плотность')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# var_smoothing влияние
smooth_vals = np.logspace(-12, -3, 30)
smooth_accs = []
for sv in smooth_vals:
    m = GaussianNB(var_smoothing=sv)
    m.fit(X_tr, y_tr)
    smooth_accs.append(accuracy_score(y_te, m.predict(X_te)))
axes[1,2].semilogx(smooth_vals, smooth_accs, color=PALETTE[2], lw=2)
axes[1,2].set_xlabel('var_smoothing (log scale)')
axes[1,2].set_ylabel('Test Accuracy')
axes[1,2].set_title('GaussianNB: влияние var_smoothing', fontweight='bold')
axes[1,2].grid(True, alpha=0.3)

plt.suptitle(' Наивный Байес — Стоматологическая клиника', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('\nНаивный Байес — результаты:')
for name, res in nb_results.items():
    print(f'  {name:<18}: Accuracy={res["acc"]:.4f}  F1={res["f1"]:.4f}')
print()
print(' GaussianNB предполагает нормальное распределение признаков.')
print('   Несмотря на «наивность», работает удивительно хорошо на медицинских данных.')

---
## Метод 5: Случайный лес (Random Forest)

### Математика
Random Forest — **ансамбль** деревьев решений, обученных на **бутстрап-выборках** (bagging).

**Bagging (Bootstrap Aggregating):**
1. Для каждого дерева $T_b$ из $B$ деревьев:
   - Выбираем случайную подвыборку $D_b$ с возвращением
   - При каждом разбиении выбираем случайное подмножество $m = \sqrt{p}$ признаков
   - Строим дерево без обрезки (глубокое)
2. Итоговый прогноз: **мажоритарное голосование** (классификация) или **среднее** (регрессия):
$$\hat{y} = \frac{1}{B}\sum_{b=1}^{B} T_b(\mathbf{x})$$

**Out-of-Bag (OOB) оценка:** ~37% данных не попадают в $D_b$ — используются для валидации без hold-out.

In [ ]:
# ── Случайный лес ─────────────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, max_depth=None,
                             max_features='sqrt', oob_score=True,
                             n_jobs=-1, random_state=42)
rf.fit(X_tr, y_tr)
y_pred_rf = rf.predict(X_te)
y_prob_rf = rf.predict_proba(X_te)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# OOB score vs n_estimators
oob_scores = []
n_trees_range = [5, 10, 20, 50, 100, 200, 300]
for n in n_trees_range:
    m = RandomForestClassifier(n_estimators=n, oob_score=True, n_jobs=-1, random_state=42)
    m.fit(X_tr, y_tr)
    oob_scores.append(m.oob_score_)
axes[0,0].plot(n_trees_range, oob_scores, 'o-', color=PALETTE[2], lw=2, ms=7)
axes[0,0].set_xlabel('n_estimators (число деревьев)')
axes[0,0].set_ylabel('OOB Score')
axes[0,0].set_title('OOB Score vs кол-во деревьев', fontweight='bold')
axes[0,0].grid(True, alpha=0.3)
print(f'OOB Score (n=200): {rf.oob_score_:.4f}')

# Матрица ошибок
cm_rf = confusion_matrix(y_te, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Purples', ax=axes[0,1],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0,1].set_title(f'RF Матрица ошибок\nAcc={accuracy_score(y_te,y_pred_rf):.3f}', fontweight='bold')
axes[0,1].tick_params(axis='x', rotation=15)

# Feature Importance (MDI)
fi_rf = rf.feature_importances_
std_fi = np.std([t.feature_importances_ for t in rf.estimators_], axis=0)
sorted_idx = np.argsort(fi_rf)
axes[0,2].barh([FEATURES[i] for i in sorted_idx], fi_rf[sorted_idx],
               xerr=std_fi[sorted_idx], color=PALETTE[2], alpha=0.85,
               edgecolor='white', capsize=4)
axes[0,2].set_title('Feature Importance (MDI ± std)', fontweight='bold')
axes[0,2].set_xlabel('Mean Decrease in Impurity')
axes[0,2].grid(True, alpha=0.3, axis='x')

# ROC-кривые (One-vs-Rest)
y_bin = label_binarize(y_te, classes=[0,1,2])
for cls_idx, (cls_name, color) in enumerate(zip(CLASS_NAMES, PALETTE)):
    fpr, tpr, _ = roc_curve(y_bin[:,cls_idx], y_prob_rf[:,cls_idx])
    roc_auc = auc(fpr, tpr)
    axes[1,0].plot(fpr, tpr, color=color, lw=2, label=f'{cls_name} (AUC={roc_auc:.3f})')
axes[1,0].plot([0,1],[0,1],'k--',lw=1,label='Случайный классификатор')
axes[1,0].set_xlabel('False Positive Rate'); axes[1,0].set_ylabel('True Positive Rate')
axes[1,0].set_title('ROC-кривые (One-vs-Rest)', fontweight='bold')
axes[1,0].legend(fontsize=8); axes[1,0].grid(True, alpha=0.3)

# Learning Curve
train_sizes, train_scores, val_scores = learning_curve(
    RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    X, y, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 8), scoring='accuracy'
)
tr_mean = train_scores.mean(axis=1)
va_mean = val_scores.mean(axis=1)
tr_std  = train_scores.std(axis=1)
va_std  = val_scores.std(axis=1)
axes[1,1].plot(train_sizes, tr_mean, 'o-', color=PALETTE[0], label='Train', lw=2)
axes[1,1].plot(train_sizes, va_mean, 's--', color=PALETTE[1], label='Validation', lw=2)
axes[1,1].fill_between(train_sizes, tr_mean-tr_std, tr_mean+tr_std, alpha=0.15, color=PALETTE[0])
axes[1,1].fill_between(train_sizes, va_mean-va_std, va_mean+va_std, alpha=0.15, color=PALETTE[1])
axes[1,1].set_xlabel('Размер обучающей выборки')
axes[1,1].set_ylabel('Accuracy')
axes[1,1].set_title('Кривая обучения (Learning Curve)', fontweight='bold')
axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

# Сравнение дерева vs леса
names_comp = ['Decision Tree\n(depth=best)', 'Random Forest\n(n=200)']
accs_comp = [accuracy_score(y_te, y_pred_dt), accuracy_score(y_te, y_pred_rf)]
f1s_comp  = [f1_score(y_te, y_pred_dt, average='macro'), f1_score(y_te, y_pred_rf, average='macro')]
x_comp = np.arange(2); w_comp = 0.35
axes[1,2].bar(x_comp-w_comp/2, accs_comp, w_comp, label='Accuracy', color=PALETTE[0], alpha=0.85)
axes[1,2].bar(x_comp+w_comp/2, f1s_comp, w_comp, label='F1-macro', color=PALETTE[2], alpha=0.85)
axes[1,2].set_xticks(x_comp); axes[1,2].set_xticklabels(names_comp)
axes[1,2].set_title('Дерево vs Случайный лес', fontweight='bold')
axes[1,2].legend(); axes[1,2].grid(True, alpha=0.3, axis='y'); axes[1,2].set_ylim(0,1.1)

plt.suptitle('🌲🌲 Случайный лес — Стоматологическая клиника', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\nRandom Forest:')
print(f'  Accuracy:  {accuracy_score(y_te, y_pred_rf):.4f}')
print(f'  OOB Score: {rf.oob_score_:.4f}')
print(f'  F1-macro:  {f1_score(y_te, y_pred_rf, average="macro"):.4f}')
print()
print(' Random Forest устойчив к переобучению за счёт усреднения множества деревьев.')

---
## Метод 6: Градиентный бустинг (Gradient Boosting)

### Математика
Gradient Boosting строит **ансамбль слабых моделей последовательно**, каждая следующая компенсирует ошибки предыдущей:

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

где $h_m$ — дерево, обученное предсказывать **отрицательный градиент потерь:**
$$r_i^{(m)} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F=F_{m-1}}$$

**Для MSE:** $r_i = y_i - F_{m-1}(x_i)$ — просто остатки!  
**Для Log Loss:** $r_i = y_i - p_i$  

Параметры: `learning_rate` (η), `n_estimators`, `max_depth`, `subsample`

In [ ]:
# ── Градиентный бустинг ───────────────────────────────────────────────────────
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                 max_depth=4, subsample=0.8, random_state=42)
gb.fit(X_tr, y_tr)
y_pred_gb = gb.predict(X_te)
y_prob_gb = gb.predict_proba(X_te)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Кривая потерь по итерациям (staged_predict)
train_losses, test_losses = [], []
from sklearn.metrics import log_loss
for y_prob_tr, y_prob_te in zip(gb.staged_predict_proba(X_tr), gb.staged_predict_proba(X_te)):
    train_losses.append(log_loss(y_tr, y_prob_tr))
    test_losses.append(log_loss(y_te, y_prob_te))
iters = range(1, len(train_losses)+1)
axes[0,0].plot(list(iters), train_losses, color=PALETTE[0], lw=2, label='Train Log Loss')
axes[0,0].plot(list(iters), test_losses,  color=PALETTE[1], lw=2, ls='--', label='Test Log Loss')
best_iter = np.argmin(test_losses)+1
axes[0,0].axvline(x=best_iter, color='red', ls=':', label=f'Лучшая итерация={best_iter}')
axes[0,0].set_xlabel('Итерация (n_estimators)')
axes[0,0].set_ylabel('Log Loss')
axes[0,0].set_title('Кривая потерь при бустинге', fontweight='bold')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# Влияние learning_rate
lr_vals = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
lr_accs = []
for lr_val in lr_vals:
    m = GradientBoostingClassifier(n_estimators=100, learning_rate=lr_val,
                                    max_depth=3, random_state=42)
    m.fit(X_tr, y_tr)
    lr_accs.append(accuracy_score(y_te, m.predict(X_te)))
axes[0,1].semilogx(lr_vals, lr_accs, 'D-', color=PALETTE[4], lw=2, ms=8)
axes[0,1].set_xlabel('learning_rate (log scale)')
axes[0,1].set_ylabel('Test Accuracy')
axes[0,1].set_title('Влияние learning rate', fontweight='bold')
axes[0,1].grid(True, alpha=0.3)

# Feature importance
fi_gb = gb.feature_importances_
sorted_idx_gb = np.argsort(fi_gb)
axes[0,2].barh([FEATURES[i] for i in sorted_idx_gb], fi_gb[sorted_idx_gb],
               color=PALETTE[4], alpha=0.85, edgecolor='white')
axes[0,2].set_title('Feature Importance (GB)', fontweight='bold')
axes[0,2].set_xlabel('Importance'); axes[0,2].grid(True, alpha=0.3, axis='x')

# Матрица ошибок
cm_gb = confusion_matrix(y_te, y_pred_gb)
sns.heatmap(cm_gb, annot=True, fmt='d', cmap='Reds', ax=axes[1,0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[1,0].set_title(f'GB Матрица ошибок\nAcc={accuracy_score(y_te,y_pred_gb):.3f}', fontweight='bold')
axes[1,0].tick_params(axis='x', rotation=15)

# Сравнение методов ансамблей: RF vs GB vs AdaBoost
ada = AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=42, algorithm='SAMME')
ada.fit(X_tr, y_tr)
y_pred_ada = ada.predict(X_te)

ensemble_names = ['Random Forest', 'Grad. Boosting', 'AdaBoost']
ensemble_preds = [y_pred_rf, y_pred_gb, y_pred_ada]
ens_accs = [accuracy_score(y_te, p) for p in ensemble_preds]
ens_f1s  = [f1_score(y_te, p, average='macro') for p in ensemble_preds]
x_ens = np.arange(3); w_ens = 0.35
axes[1,1].bar(x_ens-w_ens/2, ens_accs, w_ens, label='Accuracy', color=PALETTE[2], alpha=0.85)
axes[1,1].bar(x_ens+w_ens/2, ens_f1s, w_ens, label='F1-macro', color=PALETTE[4], alpha=0.85)
axes[1,1].set_xticks(x_ens); axes[1,1].set_xticklabels(ensemble_names)
axes[1,1].set_title('Сравнение ансамблей: RF / GB / AdaBoost', fontweight='bold')
axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3, axis='y'); axes[1,1].set_ylim(0,1.1)
for i, (a, f) in enumerate(zip(ens_accs, ens_f1s)):
    axes[1,1].text(i-w_ens/2, a+0.01, f'{a:.3f}', ha='center', fontsize=8, fontweight='bold')

# Частичная зависимость (Partial Dependence)
cost_range = np.linspace(X[:,1].min(), X[:,1].max(), 60)
X_pdp = np.tile(X.mean(axis=0), (60,1))
X_pdp[:,1] = cost_range
pdp_probs = gb.predict_proba(X_pdp)
for cls_idx, (cls_name, color) in enumerate(zip(CLASS_NAMES, PALETTE)):
    axes[1,2].plot(cost_range, pdp_probs[:,cls_idx], color=color, lw=2.5, label=cls_name)
axes[1,2].set_xlabel('Стоимость (нормализовано)')
axes[1,2].set_ylabel('P(класс | cost, остальные=среднее)')
axes[1,2].set_title('Частичная зависимость от стоимости (PDP)', fontweight='bold')
axes[1,2].legend(fontsize=8); axes[1,2].grid(True, alpha=0.3)

plt.suptitle(' Градиентный бустинг — Стоматологическая клиника', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\nGradient Boosting: Accuracy={accuracy_score(y_te, y_pred_gb):.4f}  F1={f1_score(y_te, y_pred_gb, average="macro"):.4f}')

---
## Методы 7–9: SVM, k-NN, MLP

### SVM (Support Vector Machine)
Ищет **гиперплоскость максимального зазора** (maximum margin hyperplane):
$$\min_{w,b} \frac{1}{2}||w||^2 + C\sum_i \xi_i \quad \text{s.t.} \quad y_i(w^T x_i + b) \geq 1 - \xi_i$$
**Ядро RBF:** $K(x,x') = \exp(-\gamma||x-x'||^2)$ — нелинейные границы.

### k-NN (k Nearest Neighbors)
Для новой точки находим $k$ ближайших соседей по евклидову расстоянию:
$$d(x, x') = \sqrt{\sum_{i=1}^{n}(x_i - x'_i)^2}$$
Класс — мажоритарное голосование среди $k$ соседей.

### MLP (Multilayer Perceptron)
Нейронная сеть с прямым распространением:
$$h^{(l)} = \sigma(W^{(l)} h^{(l-1)} + b^{(l)}) \quad \text{где} \quad \sigma(z) = \max(0,z) \text{ (ReLU)}$$
Обучается через **обратное распространение ошибки** (backpropagation) + Adam/SGD.

In [ ]:
# ── SVM, k-NN, MLP ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# SVM с разными ядрами
svm_kernels = {'linear': SVC(kernel='linear', C=1.0, probability=True, random_state=42),
               'rbf':    SVC(kernel='rbf', C=10.0, gamma=0.1, probability=True, random_state=42),
               'poly':   SVC(kernel='poly', degree=3, C=1.0, probability=True, random_state=42)}
svm_results = {}
for kernel_name, svm_model in svm_kernels.items():
    svm_model.fit(X_tr, y_tr)
    svm_pred = svm_model.predict(X_te)
    svm_results[kernel_name] = {
        'acc': accuracy_score(y_te, svm_pred),
        'f1':  f1_score(y_te, svm_pred, average='macro'),
        'n_sv': svm_model.n_support_.sum(),
    }

# Bar: SVM kernels
knames = list(svm_results.keys())
sv_accs = [svm_results[k]['acc'] for k in knames]
sv_f1s  = [svm_results[k]['f1']  for k in knames]
sv_nsv  = [svm_results[k]['n_sv'] for k in knames]
x_sv = np.arange(3); w_sv = 0.25
axes[0,0].bar(x_sv-w_sv, sv_accs, w_sv, label='Accuracy', color=PALETTE[0], alpha=0.85)
axes[0,0].bar(x_sv,      sv_f1s,  w_sv, label='F1-macro',  color=PALETTE[1], alpha=0.85)
axes[0,0].bar(x_sv+w_sv, [n/500 for n in sv_nsv], w_sv, label='SV/500', color=PALETTE[2], alpha=0.85)
axes[0,0].set_xticks(x_sv); axes[0,0].set_xticklabels(knames)
axes[0,0].set_title('SVM: сравнение ядер', fontweight='bold')
axes[0,0].legend(fontsize=8); axes[0,0].grid(True, alpha=0.3, axis='y')

# k-NN: влияние k
k_vals = range(1, 31)
knn_tr_accs, knn_te_accs = [], []
for k_val in k_vals:
    knn = KNeighborsClassifier(n_neighbors=k_val, metric='euclidean', weights='uniform')
    knn.fit(X_tr, y_tr)
    knn_tr_accs.append(accuracy_score(y_tr, knn.predict(X_tr)))
    knn_te_accs.append(accuracy_score(y_te, knn.predict(X_te)))
best_k = list(k_vals)[np.argmax(knn_te_accs)]
axes[0,1].plot(list(k_vals), knn_tr_accs, 'o-', color=PALETTE[0], lw=2, ms=4, label='Train')
axes[0,1].plot(list(k_vals), knn_te_accs, 's--', color=PALETTE[1], lw=2, ms=4, label='Test')
axes[0,1].axvline(x=best_k, color='red', ls=':', label=f'Лучший k={best_k}')
axes[0,1].set_xlabel('k (число соседей)')
axes[0,1].set_ylabel('Accuracy')
axes[0,1].set_title('k-NN: влияние k', fontweight='bold')
axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

# MLP: архитектуры
mlp_architectures = {
    '(64,)':         MLPClassifier(hidden_layer_sizes=(64,), activation='relu', max_iter=500, random_state=42),
    '(128, 64)':     MLPClassifier(hidden_layer_sizes=(128,64), activation='relu', max_iter=500, random_state=42),
    '(256,128,64)':  MLPClassifier(hidden_layer_sizes=(256,128,64), activation='relu', max_iter=500, random_state=42),
    'tanh (128,64)': MLPClassifier(hidden_layer_sizes=(128,64), activation='tanh', max_iter=500, random_state=42),
}
mlp_results = {}
for arch, mlp_model in mlp_architectures.items():
    mlp_model.fit(X_tr, y_tr)
    mlp_pred = mlp_model.predict(X_te)
    mlp_results[arch] = {
        'acc': accuracy_score(y_te, mlp_pred),
        'f1':  f1_score(y_te, mlp_pred, average='macro'),
        'loss_curve': mlp_model.loss_curve_,
    }

# MLP loss curves
for (arch, res), color in zip(mlp_results.items(), PALETTE):
    axes[0,2].plot(res['loss_curve'], color=color, lw=2, label=f'{arch} (Acc={res["acc"]:.3f})')
axes[0,2].set_xlabel('Эпоха'); axes[0,2].set_ylabel('Loss')
axes[0,2].set_title('MLP: кривые потерь разных архитектур', fontweight='bold')
axes[0,2].legend(fontsize=7); axes[0,2].grid(True, alpha=0.3)

# Матрица ошибок лучшего SVM (rbf)
svm_rbf = SVC(kernel='rbf', C=10.0, gamma=0.1, probability=True, random_state=42)
svm_rbf.fit(X_tr, y_tr)
y_pred_svm = svm_rbf.predict(X_te)
cm_svm = confusion_matrix(y_te, y_pred_svm)
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1,0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[1,0].set_title(f'SVM (RBF) Матрица ошибок\nAcc={accuracy_score(y_te,y_pred_svm):.3f}', fontweight='bold')
axes[1,0].tick_params(axis='x', rotation=15)

# k-NN граница решения
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X2_tr, y2_tr)
Z_knn = knn_best.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[1,1].contourf(xx, yy, Z_knn, alpha=0.25, cmap='Set2')
for cls in [0,1,2]:
    mask = y2_te == cls
    axes[1,1].scatter(X2_te[mask,0], X2_te[mask,1], c=PALETTE[cls],
                     s=30, alpha=0.8, edgecolors='white', lw=0.3, label=CLASS_NAMES[cls])
axes[1,1].set_title(f'k-NN (k={best_k}) граница решения', fontweight='bold')
axes[1,1].legend(fontsize=8); axes[1,1].grid(True, alpha=0.2)

# MLP Bar сравнение
arch_names = list(mlp_results.keys())
mlp_accs = [mlp_results[a]['acc'] for a in arch_names]
mlp_f1s  = [mlp_results[a]['f1']  for a in arch_names]
x_mlp = np.arange(len(arch_names)); w_mlp = 0.35
axes[1,2].bar(x_mlp-w_mlp/2, mlp_accs, w_mlp, label='Accuracy', color=PALETTE[5], alpha=0.85)
axes[1,2].bar(x_mlp+w_mlp/2, mlp_f1s, w_mlp, label='F1-macro', color=PALETTE[6], alpha=0.85)
axes[1,2].set_xticks(x_mlp); axes[1,2].set_xticklabels(arch_names, rotation=15, fontsize=8)
axes[1,2].set_title('MLP: сравнение архитектур', fontweight='bold')
axes[1,2].legend(); axes[1,2].grid(True, alpha=0.3, axis='y'); axes[1,2].set_ylim(0,1.1)

plt.suptitle(' SVM · k-NN · MLP — Стоматологическая клиника', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

knn_final = KNeighborsClassifier(n_neighbors=best_k)
knn_final.fit(X_tr, y_tr)
y_pred_knn = knn_final.predict(X_te)
mlp_final = MLPClassifier(hidden_layer_sizes=(128,64), activation='relu', max_iter=500, random_state=42)
mlp_final.fit(X_tr, y_tr)
y_pred_mlp = mlp_final.predict(X_te)

print(f'SVM (RBF):  Acc={accuracy_score(y_te, y_pred_svm):.4f}  F1={f1_score(y_te, y_pred_svm, average="macro"):.4f}')
print(f'k-NN (k={best_k}): Acc={accuracy_score(y_te, y_pred_knn):.4f}  F1={f1_score(y_te, y_pred_knn, average="macro"):.4f}')
print(f'MLP (128,64):Acc={accuracy_score(y_te, y_pred_mlp):.4f}  F1={f1_score(y_te, y_pred_mlp, average="macro"):.4f}')

---
## 📊 Метод 10: Итоговое сравнение всех классификаторов

In [ ]:
# ── Все модели ────────────────────────────────────────────────────────────────
all_models = {
    'Logistic Reg.':    (lr,                 X_tr, X_te),
    'Decision Tree':    (dt_best,             X_tr, X_te),
    'Naive Bayes':      (GaussianNB(),        X_tr, X_te),
    'Random Forest':    (rf,                  X_tr, X_te),
    'Grad. Boosting':   (gb,                  X_tr, X_te),
    'AdaBoost':         (ada,                 X_tr, X_te),
    'SVM (RBF)':        (svm_rbf,             X_tr, X_te),
    'k-NN':             (knn_final,           X_tr, X_te),
    'MLP (128,64)':     (mlp_final,           X_tr, X_te),
}

results_all = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, (model, Xtr, Xte) in all_models.items():
    if not hasattr(model, 'predict') or name in ['Logistic Reg.','Decision Tree','Random Forest',
                                                   'Grad. Boosting','AdaBoost','SVM (RBF)','k-NN','MLP (128,64)']:
        pass
    else:
        model.fit(Xtr, y_tr)
    
    if name == 'Naive Bayes':
        model.fit(Xtr, y_tr)
    
    pred = model.predict(Xte)
    cv_scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy', n_jobs=-1)
    
    results_all.append({
        'Модель':     name,
        'Accuracy':   round(accuracy_score(y_te, pred), 4),
        'F1-macro':   round(f1_score(y_te, pred, average='macro'), 4),
        'Precision':  round(precision_score(y_te, pred, average='macro'), 4),
        'Recall':     round(recall_score(y_te, pred, average='macro'), 4),
        'CV mean':    round(cv_scores.mean(), 4),
        'CV std':     round(cv_scores.std(), 4),
    })

df_all = pd.DataFrame(results_all).sort_values('Accuracy', ascending=False).reset_index(drop=True)

print('='*75)
print('ИТОГОВОЕ СРАВНЕНИЕ ВСЕХ КЛАССИФИКАТОРОВ')
print('='*75)
print(df_all.to_string(index=False))

# ── Визуализация ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Accuracy + F1 bar chart
x_all = np.arange(len(df_all)); w_all = 0.35
colors_all = [PALETTE[i%len(PALETTE)] for i in range(len(df_all))]
bars_acc = axes[0,0].bar(x_all-w_all/2, df_all['Accuracy'], w_all,
                          color=[PALETTE[0]]*len(df_all), alpha=0.85, label='Accuracy')
bars_f1  = axes[0,0].bar(x_all+w_all/2, df_all['F1-macro'], w_all,
                          color=[PALETTE[2]]*len(df_all), alpha=0.85, label='F1-macro')
axes[0,0].set_xticks(x_all)
axes[0,0].set_xticklabels(df_all['Модель'], rotation=35, ha='right', fontsize=8)
axes[0,0].set_title('Accuracy и F1-macro всех моделей', fontweight='bold')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3, axis='y')
axes[0,0].set_ylim(0, 1.1)
for bar, val in zip(bars_acc, df_all['Accuracy']):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                  f'{val:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

# 2. CV boxplot
cv_data_all = []
for name, (model, Xtr, Xte) in all_models.items():
    scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy', n_jobs=-1)
    cv_data_all.append(scores)
bp = axes[0,1].boxplot(cv_data_all, labels=list(all_models.keys()),
                        patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], PALETTE*2):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[0,1].set_xticklabels(list(all_models.keys()), rotation=35, ha='right', fontsize=8)
axes[0,1].set_title('5-fold CV: распределение Accuracy', fontweight='bold')
axes[0,1].set_ylabel('Accuracy'); axes[0,1].grid(True, alpha=0.3, axis='y')

# 3. ROC multi-model (для лучшей модели — Random Forest)
y_bin_all = label_binarize(y_te, classes=[0,1,2])
best_model = rf  # Random Forest
best_probs = best_model.predict_proba(X_te)
for cls_idx, (cls_name, color) in enumerate(zip(CLASS_NAMES, PALETTE)):
    fpr, tpr, _ = roc_curve(y_bin_all[:,cls_idx], best_probs[:,cls_idx])
    roc_auc_val = auc(fpr, tpr)
    axes[1,0].plot(fpr, tpr, color=color, lw=2.5, label=f'{cls_name} AUC={roc_auc_val:.3f}')
axes[1,0].plot([0,1],[0,1],'k--',lw=1); axes[1,0].fill_between([0,1],[0,1], alpha=0.05, color='gray')
axes[1,0].set_xlabel('FPR'); axes[1,0].set_ylabel('TPR')
axes[1,0].set_title('ROC-кривые лучшей модели (Random Forest)', fontweight='bold')
axes[1,0].legend(fontsize=9); axes[1,0].grid(True, alpha=0.3)

# 4. Heatmap: Precision/Recall/F1 по классам (лучшая модель)
from sklearn.metrics import precision_recall_fscore_support
prec, rec, f1_per, _ = precision_recall_fscore_support(y_te, rf.predict(X_te))
pr_df = pd.DataFrame({'Precision': prec, 'Recall': rec, 'F1': f1_per},
                      index=CLASS_NAMES)
sns.heatmap(pr_df, annot=True, fmt='.3f', cmap='YlGn', ax=axes[1,1],
            cbar_kws={'shrink':0.8}, vmin=0.7, vmax=1.0)
axes[1,1].set_title('Precision / Recall / F1 по классам (RF)', fontweight='bold')

plt.suptitle('📊 Итоговое сравнение всех классификаторов\n🦷 Стоматологическая клиника',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('classifiers_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n Лучший классификатор: {df_all.iloc[0]["Модель"]} (Accuracy={df_all.iloc[0]["Accuracy"]})')
print(f'   CV Score: {df_all.iloc[0]["CV mean"]} ± {df_all.iloc[0]["CV std"]}')
print()
print(' Рекомендации для стоматологической клиники:')
print('   • Random Forest/GradBoost — лучшие по точности (production)')
print('   • Decision Tree — интерпретируемые правила для врачей')
print('   • Logistic Reg. — быстрое базовое решение + вероятности')
print('   • Naive Bayes — если данных мало (малая клиника)')